<a href="https://colab.research.google.com/github/mcristinasanchez-ai/TFG_MATEM-TICAS_M.CristinaS-nchezTim-n/blob/main/PSO_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

w=0.1 c1=1.19 c2=1.9

In [ ]:
import numpy as np
import pandas as pd

# 1. Funciones objetivo que vamos a minimizar
def sphere(x):
    return np.sum(x**2)

def schwefel_1(x):
    return np.sum(np.abs(x)) + np.prod(np.abs(x))

def schwefel_2(x):
    return np.sum([np.sum(x[:i+1])**2 for i in range(len(x))])

def schwefel_3(x):
    return np.max(np.abs(x))

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (x[:-1] - 1.0)**2.0)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def quartic(x):
    return np.sum(np.arange(1, len(x) + 1) * (x**4)) + np.random.random()

def schwefel_4(x):
    return np.sum(-x * np.sin(np.sqrt(np.abs(x))))

def rastrigin(x):
    return np.sum(x**2 - 10 * np.cos(2 * np.pi * x) + 10)

def ackley(x):
    n = float(len(x))
    return -20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n)) - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.exp(1)


funciones_prueba = {
    "Sphere": (sphere, -100, 100),
    "Schwefel 1": (schwefel_1, -10, 10),
    "Schwefel 2": (schwefel_2, -100, 100),
    "Schwefel 3": (schwefel_3, -100, 100),
    "Rosenbrock": (rosenbrock, -30, 30),
    "Step": (step, -100, 100),
    "Quartic": (quartic, -1.28, 1.28),
    "Schwefel 4": (schwefel_4, -500, 500),
    "Rastrigin": (rastrigin, -5.12, 5.12),
    "Ackley": (ackley, -32, 32)
}

# 2. Parámetros
num_particulas = 100 #cantidad de partículas en el espacio de búsqueda
num_iteraciones = 10000  #número pasos máximos que dara el enjambre de partículas en cada intento
dimension = 30 #espacio de búsqueda de 30 dimensiones
w = 0.1 #coeficiente de inercia que controla la tendencia de la partícula a seguir es su dirección actual
c1 = 1.9 #coeficiente cognitivo: cuanto confía la partícula en su propia experiencia
c2 = 1.9 #coeficiente social: cuanto influye el grupo en la partícula

num_ejecuciones = 10 #Numero de ejecuciones

#Se Almacenan los mejores resultados de cada ejecución
resultados_totales = {}

print("Iniciando optimizaciones...")

for nombre_funcion, (f, limite_inf, limite_sup) in funciones_prueba.items():
    print(f"\nEjecutando 10 veces para: {nombre_funcion}...")
    resultados_funcion = []  # Almacena los 10 mejores globales de esta función

    for ejecucion in range(num_ejecuciones):
        # 3. Inicialización
        posicion = np.random.uniform(limite_inf, limite_sup, (num_particulas, dimension)) #matriz donde se colocan las 100 partículas de forma aleatoria
        velocidad = np.zeros((num_particulas, dimension))#se inicializa la velocidad a 0, las partículas inicialmente estan paradas

        pmejor_posicion = posicion.copy() #se guarda la mejor posición de la partícula
        pmejor_valor = np.array([f(p) for p in posicion]) #se calcula su valor
        gmejor_posicion = pmejor_posicion[np.argmin(pmejor_valor)].copy() #se guarda la mejor posción del enjambre
        gmejor_valor = np.min(pmejor_valor) #se calcula su valor

        # 4. Bucle principal del algoritmo PSO
        for _ in range(num_iteraciones):

            #Se generan r1 y r2 aleatorios para posteriormente actualizar la velocidad
            #y la posición
            r1 = np.random.rand(num_particulas, dimension)
            r2 = np.random.rand(num_particulas, dimension)

            velocidad = (w * velocidad +
                         c1 * r1 * (pmejor_posicion - posicion) +
                         c2 * r2 * (gmejor_posicion - posicion))

            posicion += velocidad
            #Se usa np.clip para que las partículas no busquen en zonas no válidas
            #si se superan los límites permitidos
            posicion = np.clip(posicion, limite_inf, limite_sup)

            for i in range(num_particulas):
                valor_actual = f(posicion[i])
                if valor_actual < pmejor_valor[i]:
                    pmejor_valor[i] = valor_actual
                    pmejor_posicion[i] = posicion[i].copy()

                    if valor_actual < gmejor_valor:
                        gmejor_valor = valor_actual
                        gmejor_posicion = posicion[i].copy()

        resultados_funcion.append(gmejor_valor)
        print(f"  Ejecución {ejecucion + 1}: {gmejor_valor:.4e}")

    resultados_totales[nombre_funcion] = resultados_funcion


#CREAMOS LA TABLA CON LOS RESULTADOS DE CADA FUNCIÓN EN LAS 10 Iteraciones y
#Y CALCULAMOS LA MEDIA, MEDIANA Y VARIANZA


# 1. Se crea el Dataframe donde cada columna es una de las funciones de prueba y
#y cada fila son las ejecuiones
df_inicial = pd.DataFrame(resultados_totales)
df_inicial.index = [f"EJECUCIÓN {i+1}" for i in range(num_ejecuciones)]
df_final = df_inicial.copy()

# 2. Se calcula la media, mediana y varianza
df_final.loc['MEDIA'] = df_inicial.mean(axis=0)
df_final.loc['MEDIANA'] = df_inicial.median(axis=0)
df_final.loc['VARIANZA'] = df_inicial.var(axis=0)

# 3. Se muestan los resultados por pantalla y se guardan los resultados en la
#hoja de cáculo
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL")
print("="*50)
pd.set_option('display.float_format', lambda x: f'{x:.4e}')
print(df_final)
df_final.to_excel("Tabla comparativaPSO2.xlsx")

Iniciando optimizaciones...

Ejecutando 10 veces para: Sphere...
  Ejecución 1: 4.8323e+03
  Ejecución 2: 3.2209e+03
  Ejecución 3: 8.6652e+02
  Ejecución 4: 8.1982e+01
  Ejecución 5: 4.6200e+02
  Ejecución 6: 1.0861e+03
  Ejecución 7: 2.5584e+03
  Ejecución 8: 1.9295e+03
  Ejecución 9: 1.2001e+03
  Ejecución 10: 3.6450e+03

Ejecutando 10 veces para: Schwefel 1...
  Ejecución 1: 5.2983e+01
  Ejecución 2: 2.0001e+01
  Ejecución 3: 2.5621e+01
  Ejecución 4: 4.7256e+01
  Ejecución 5: 5.1092e+01
  Ejecución 6: 4.8040e+01
  Ejecución 7: 1.7790e+01
  Ejecución 8: 5.5966e+01
  Ejecución 9: 3.7783e+01
  Ejecución 10: 1.4407e+01

Ejecutando 10 veces para: Schwefel 2...
  Ejecución 1: 5.0000e+03
  Ejecución 2: 6.6667e+03
  Ejecución 3: 1.0000e+04
  Ejecución 4: 5.0000e+03
  Ejecución 5: 1.0000e+04
  Ejecución 6: 2.1351e-20
  Ejecución 7: 1.4552e-20
  Ejecución 8: 1.0000e+04
  Ejecución 9: 3.7138e-21
  Ejecución 10: 3.4289e-20

Ejecutando 10 veces para: Schwefel 3...
  Ejecución 1: 9.0072e-11
  E